# Python for Health, Social, and Economic Data Science: Lecture Five


## Lecture Focus

In this lecture we model applied outcomes that matter in practice:
- health demand (emergency admissions)
- social risk (school absence and deprivation)
- economic pressure (income and unemployment)

We will use `statsmodels` for interpretable regression, `scikit-learn` for prediction and segmentation, and lightweight text modelling for social-care notes.


## 17. StatsModels for Health, Social, and Economic Outcomes


### 17.1 Imports and Reproducibility


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    mean_squared_error,
    r2_score,
    precision_score,
    recall_score,
)
from sklearn.cluster import KMeans

rng = np.random.default_rng(42)
sns.set_theme(style='whitegrid')


### 17.2 Build a Synthetic Policy Dataset


In [ ]:
areas = [f"Area_{i:02d}" for i in range(1, 21)]
records = []

for area in areas:
    base_income = rng.normal(32000, 4500)
    deprivation = np.clip(rng.normal(50, 15), 15, 90)

    for month in range(1, 13):
        unemployment_rate = np.clip(rng.normal(0.07 + (deprivation - 50) / 700, 0.015), 0.03, 0.20)
        school_absence_rate = np.clip(rng.normal(0.055 + deprivation / 1500, 0.01), 0.02, 0.16)
        chronic_rate = np.clip(rng.normal(0.16 + deprivation / 500, 0.02), 0.08, 0.45)
        preventive_visits = np.clip(rng.normal(230 - 1.1 * deprivation + (base_income - 32000) / 150, 20), 110, 340)
        social_support_spend = np.clip(rng.normal(1_850_000 - 7_000 * deprivation, 140_000), 850_000, 2_500_000)
        median_income = np.clip(rng.normal(base_income, 1200), 18000, 50000)
        winter = 1 if month in [11, 12, 1, 2] else 0

        emergency_admissions = (
            2.2
            + 21 * unemployment_rate
            + 14 * chronic_rate
            + 17 * school_absence_rate
            - 0.0055 * preventive_visits
            - 0.0000009 * social_support_spend
            + 0.85 * winter
            + rng.normal(0, 0.8)
        )

        records.append({
            'area': area,
            'month': month,
            'median_income': median_income,
            'deprivation_index': deprivation,
            'unemployment_rate': unemployment_rate,
            'school_absence_rate': school_absence_rate,
            'chronic_rate': chronic_rate,
            'preventive_visits': preventive_visits,
            'social_support_spend': social_support_spend,
            'winter': winter,
            'emergency_admissions_per_1k': max(emergency_admissions, 0.4),
        })

health_df = pd.DataFrame(records)
health_df['high_risk'] = (
    health_df['emergency_admissions_per_1k']
    > health_df['emergency_admissions_per_1k'].quantile(0.70)
).astype(int)

health_df.head()


### Quick Check: Shape and Core Variables


In [ ]:
print('Rows:', len(health_df))
print('Areas:', health_df['area'].nunique())
print('High-risk share:', round(health_df['high_risk'].mean(), 3))

health_df[['emergency_admissions_per_1k', 'unemployment_rate', 'school_absence_rate', 'median_income']].describe().T


### 17.3 OLS Modelling: Drivers of Emergency Admissions


In [ ]:
ols_formula = (
    'emergency_admissions_per_1k ~ unemployment_rate + chronic_rate + '
    'school_absence_rate + preventive_visits + social_support_spend + '
    'winter + median_income + deprivation_index'
)

ols_model = smf.ols(ols_formula, data=health_df).fit()
print(ols_model.summary())


Interpretation tip:
- positive coefficients increase expected admissions
- negative coefficients reduce expected admissions
- `P>|t|` helps assess which predictors are statistically important


### Extract and Sort Coefficients


In [ ]:
coef_table = pd.DataFrame({
    'coef': ols_model.params,
    'p_value': ols_model.pvalues
}).sort_values('coef', ascending=False)

coef_table


### 17.4 Residual Diagnostics


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

sns.residplot(
    x=ols_model.fittedvalues,
    y=ols_model.resid,
    lowess=True,
    line_kws={'color': 'red', 'lw': 2},
    ax=axs[0],
)
axs[0].set_title('Residuals vs Fitted')
axs[0].set_xlabel('Fitted admissions per 1k')
axs[0].set_ylabel('Residuals')

sm.qqplot(ols_model.resid, line='45', ax=axs[1])
axs[1].set_title('Q-Q Plot of Residuals')

plt.tight_layout()
plt.show()


### 17.5 Scenario Prediction for Policy Discussion


In [ ]:
scenario_df = pd.DataFrame({
    'unemployment_rate': [0.14, 0.14, 0.09],
    'chronic_rate': [0.25, 0.25, 0.20],
    'school_absence_rate': [0.11, 0.11, 0.07],
    'preventive_visits': [150, 220, 260],
    'social_support_spend': [1_100_000, 1_500_000, 1_950_000],
    'winter': [1, 1, 0],
    'median_income': [24000, 26000, 32000],
    'deprivation_index': [78, 72, 50],
}, index=['High-stress baseline', 'Moderate intervention', 'Stronger intervention'])

scenario_df['predicted_admissions_per_1k'] = ols_model.predict(scenario_df)
scenario_df[['predicted_admissions_per_1k']]


### 17.6 Model Evaluation and Comparison


In [ ]:
mse = mean_squared_error(health_df['emergency_admissions_per_1k'], ols_model.fittedvalues)
r2 = r2_score(health_df['emergency_admissions_per_1k'], ols_model.fittedvalues)
print(f'MSE: {mse:.3f}')
print(f'R-squared: {r2:.3f}')


In [ ]:
reduced_model = smf.ols(
    'emergency_admissions_per_1k ~ unemployment_rate + chronic_rate + school_absence_rate + winter',
    data=health_df,
).fit()

comparison = pd.DataFrame({
    'Model': ['Full model', 'Reduced model'],
    'AIC': [ols_model.aic, reduced_model.aic],
    'BIC': [ols_model.bic, reduced_model.bic],
    'R2': [ols_model.rsquared, reduced_model.rsquared],
})
comparison


### 17.7 Robust Standard Errors (HC3)


In [ ]:
robust_results = ols_model.get_robustcov_results(cov_type='HC3')
print(robust_results.summary())


## 18. Scikit-learn for Risk Prediction and Segmentation


### 18.1 Prepare Features and Train/Test Split


In [ ]:
feature_cols = [
    'unemployment_rate',
    'chronic_rate',
    'school_absence_rate',
    'preventive_visits',
    'social_support_spend',
    'median_income',
    'deprivation_index',
    'winter',
]

X = health_df[feature_cols]
y = health_df['high_risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print('Train size:', X_train.shape, 'Test size:', X_test.shape)


### 18.2 Standardisation + Logistic Regression


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logreg_model = LogisticRegression(max_iter=200, random_state=42)
logreg_model.fit(X_train_scaled, y_train)


### 18.3 Evaluate Logistic Regression


In [ ]:
y_pred_logreg = logreg_model.predict(X_test_scaled)
y_prob_logreg = logreg_model.predict_proba(X_test_scaled)[:, 1]

print(f'Accuracy: {accuracy_score(y_test, y_pred_logreg):.3f}')
print(f'ROC AUC: {roc_auc_score(y_test, y_prob_logreg):.3f}')
print('\nClassification report:')
print(classification_report(y_test, y_pred_logreg, zero_division=0))

cm = confusion_matrix(y_test, y_pred_logreg)
pd.DataFrame(cm, index=['Actual 0', 'Actual 1'], columns=['Pred 0', 'Pred 1'])


### 18.4 Random Forest Comparison


In [ ]:
rf_model = RandomForestClassifier(n_estimators=300, random_state=42, max_depth=6)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print(f'Accuracy: {accuracy_score(y_test, y_pred_rf):.3f}')
print(f'ROC AUC: {roc_auc_score(y_test, y_prob_rf):.3f}')
print('\nClassification report:')
print(classification_report(y_test, y_pred_rf, zero_division=0))


### 18.5 Model Comparison Table


In [ ]:
model_compare = pd.DataFrame({
    'model': ['LogisticRegression', 'RandomForest'],
    'accuracy': [accuracy_score(y_test, y_pred_logreg), accuracy_score(y_test, y_pred_rf)],
    'roc_auc': [roc_auc_score(y_test, y_prob_logreg), roc_auc_score(y_test, y_prob_rf)],
})
model_compare


### 18.6 Threshold Sensitivity (Policy Triage View)


In [ ]:
thresholds = [0.30, 0.40, 0.50, 0.60]
rows = []
for t in thresholds:
    y_hat_t = (y_prob_logreg >= t).astype(int)
    rows.append({
        'threshold': t,
        'precision': precision_score(y_test, y_hat_t, zero_division=0),
        'recall': recall_score(y_test, y_hat_t, zero_division=0),
    })

pd.DataFrame(rows)


### 18.7 Feature Importance from Random Forest


In [ ]:
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_,
}).sort_values('importance', ascending=False)

importance_df


### 18.8 Unsupervised Segmentation of Areas


In [ ]:
area_profile = health_df.groupby('area', as_index=False)[[
    'emergency_admissions_per_1k',
    'unemployment_rate',
    'school_absence_rate',
    'median_income',
    'preventive_visits',
    'deprivation_index',
]].mean()

cluster_features = area_profile[[
    'emergency_admissions_per_1k',
    'unemployment_rate',
    'school_absence_rate',
    'median_income',
    'deprivation_index',
]]

kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
area_profile['cluster'] = kmeans.fit_predict(cluster_features)
area_profile.sort_values(['cluster', 'emergency_admissions_per_1k']).head(12)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(
    data=area_profile,
    x='deprivation_index',
    y='emergency_admissions_per_1k',
    hue='cluster',
    size='median_income',
    palette='Set2',
    ax=ax,
)
ax.set_title('Area Segments: Deprivation vs Emergency Admissions')
ax.set_xlabel('Average deprivation index')
ax.set_ylabel('Average emergency admissions per 1k')
plt.tight_layout()
plt.show()


## 19. Text Modelling for Social and Public Health Case Notes


This section demonstrates lightweight NLP for social-data science.
We focus on triage-relevant signals from short case notes.


### 19.1 Example Corpus


In [ ]:
case_notes = [
    'Client missed two GP appointments and reports rent arrears stress.',
    'Household income dropped after job loss; school attendance concerns rising.',
    'Regular medication pickup and stable work schedule this month.',
    'Frequent emergency visits linked to uncontrolled asthma and damp housing.',
    'Community nurse reports improved mobility and better diet adherence.',
    'Family requested food-bank referral after benefits sanction.',
    'No acute symptoms; preventive screening completed on time.',
    'Social worker flags isolation, anxiety, and repeated missed appointments.',
]

pd.DataFrame({'note': case_notes})


### 19.2 Tokenization and Cleaning


In [ ]:
from nltk.tokenize import wordpunct_tokenize
from nltk.stem import PorterStemmer
from nltk import FreqDist
from nltk.collocations import BigramCollocationFinder, BigramAssocMeasures

custom_stopwords = {
    'and', 'the', 'to', 'after', 'this', 'on', 'no', 'two', 'reports', 'report',
    'client', 'family', 'worker', 'month'
}

tokens = []
for note in case_notes:
    words = [w.lower() for w in wordpunct_tokenize(note) if w.isalpha()]
    words = [w for w in words if w not in custom_stopwords]
    tokens.extend(words)

tokens[:30]


### 19.3 Most Frequent Risk Terms


In [ ]:
freq = FreqDist(tokens)
freq.most_common(15)


### 19.4 Stemmed Theme Counts


In [ ]:
stemmer = PorterStemmer()
stems = [stemmer.stem(tok) for tok in tokens]

pd.DataFrame(FreqDist(stems).most_common(12), columns=['stem', 'count'])


### 19.5 Collocations (Word Pairs)


In [ ]:
finder = BigramCollocationFinder.from_words(tokens)
collocations = finder.nbest(BigramAssocMeasures.likelihood_ratio, 10)
collocations


### 19.6 Simple Rule-Based Triage Score


In [ ]:
risk_keywords = {
    'missed': 2,
    'arrears': 2,
    'loss': 1,
    'stress': 1,
    'emergency': 2,
    'asthma': 1,
    'damp': 1,
    'sanction': 2,
    'isolation': 2,
    'anxiety': 1,
}

def score_note(note):
    words = [w.lower() for w in wordpunct_tokenize(note) if w.isalpha()]
    return sum(risk_keywords.get(w, 0) for w in words)

triage_df = pd.DataFrame({'note': case_notes})
triage_df['risk_score'] = triage_df['note'].apply(score_note)
triage_df['priority'] = np.where(triage_df['risk_score'] >= 4, 'high', 'routine')
triage_df.sort_values('risk_score', ascending=False)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(len(triage_df)), triage_df['risk_score'])
ax.set_xticks(range(len(triage_df)))
ax.set_xticklabels([f'Note {i+1}' for i in range(len(triage_df))], rotation=45, ha='right')
ax.set_ylabel('Risk score')
ax.set_title('Triage Risk Scores from Social/Health Notes')
plt.tight_layout()
plt.show()


## 20. Your Projects


Choose one modelling mini-project:
1. Estimate and compare emergency-admission models for two policy scenarios.
2. Build a classifier for high social risk and justify the chosen threshold.
3. Segment areas into policy clusters and propose targeted interventions.
4. Extend the triage note scorer with a clearer labeling strategy.


## 21. Further study (and potentially a followup class!)


Suggested next steps:
- Add fixed effects or mixed models for area-level heterogeneity.
- Introduce cross-validation and calibration for classification models.
- Move from rule-based text scoring to supervised NLP models on labeled notes.
